In [17]:
import os
import sys
from pathlib import Path
import torch

REPO_ROOT = "/vol/biomedic3/tx1215/REPA-E"
FAILURE_DIR = "/vol/biomedic3/tx1215/REPA-E/checkpoints/embed_latent_dit_sits2_256x192_debug/failures/step_0037386"
DEVICE = torch.device("cuda:0")

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from models.sit import SiT_models

def load_pt(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

ckpt = load_pt(Path(FAILURE_DIR) / "model_before_failure.pt")
rank0 = load_pt(Path(FAILURE_DIR) / "rank_0.pt")
rank1 = load_pt(Path(FAILURE_DIR) / "rank_1.pt")
args = ckpt["args"]

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("failing step:", ckpt.get("failing_step"))
print("rank0 bad gradients:", rank0.get("bad_grad_stats"))
print("rank1 bad gradients:", rank1.get("bad_grad_stats"))

PyTorch: 2.3.1+cu121
CUDA: 12.1
GPU: NVIDIA GeForce RTX 3090
failing step: 37386
rank0 bad gradients: [{'name': 'blocks.7.attn.qkv.weight', 'shape': (1152, 384), 'dtype': 'torch.float32', 'finite_absmax': 30496.0, 'num_nan': 0, 'num_posinf': 2, 'num_neginf': 1, 'numel': 442368}]
rank1 bad gradients: [{'name': 'blocks.7.attn.qkv.weight', 'shape': (1152, 384), 'dtype': 'torch.float32', 'finite_absmax': 30496.0, 'num_nan': 0, 'num_posinf': 2, 'num_neginf': 1, 'numel': 442368}]


In [18]:
def getarg(name, default=None):
    return getattr(args, name, default)

def move_to(x):
    if torch.is_tensor(x):
        return x.to(DEVICE)
    if isinstance(x, dict):
        return {k: move_to(v) for k, v in x.items()}
    if isinstance(x, list):
        return [move_to(v) for v in x]
    if isinstance(x, tuple):
        return tuple(move_to(v) for v in x)
    return x

def first_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, (list, tuple)):
        for v in x:
            t = first_tensor(v)
            if t is not None:
                return t
    if isinstance(x, dict):
        for v in x.values():
            t = first_tensor(v)
            if t is not None:
                return t
    return None

def stats(x):
    x = x.detach()
    finite = torch.isfinite(x)
    xf = x[finite].float()
    return {
        "shape": tuple(x.shape),
        "rms": torch.sqrt((xf ** 2).mean()).item() if xf.numel() else float("nan"),
        "std": xf.std(unbiased=False).item() if xf.numel() else float("nan"),
        "absmax": xf.abs().max().item() if xf.numel() else float("nan"),
        "nan": torch.isnan(x).sum().item(),
        "inf": torch.isinf(x).sum().item(),
    }

def restore_rng(snapshot):
    state = snapshot.get("forward_rng_state")
    if state is None:
        return
    if state.get("cpu") is not None:
        torch.set_rng_state(state["cpu"])
    if state.get("cuda") is not None:
        torch.cuda.set_rng_state(state["cuda"], device=DEVICE)

def precision_context(snapshot):
    precision = snapshot.get("mixed_precision", getarg("mixed_precision", "bf16"))
    if precision == "bf16":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    if precision == "fp16":
        return torch.autocast("cuda", dtype=torch.float16)
    return torch.autocast("cuda", enabled=False)

def infer_z_dims(state):
    z_dims = []
    i = 0
    while f"projectors.{i}.4.weight" in state:
        z_dims.append(state[f"projectors.{i}.4.weight"].shape[0])
        i += 1
    return z_dims

In [19]:
def build_model(snapshot):
    z = snapshot["z"]
    latent_size = tuple(z.shape[-2:])
    in_channels = z.shape[1]
    state = ckpt["model"]

    model = SiT_models[getarg("model")](
        input_size=latent_size,
        in_channels=in_channels,
        num_classes=getarg("num_classes", 1000),
        class_dropout_prob=getarg("cfg_prob", 0.1),
        z_dims=infer_z_dims(state),
        encoder_depth=getarg("encoder_depth", 8),
        bn_momentum=getarg("bn_momentum", 0.1),
        fused_attn=snapshot.get("fused_attn", getarg("fused_attn", True)),
        qk_norm=snapshot.get("qk_norm", getarg("qk_norm", False)),
    )

    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.train()
    model.bn.eval()
    return model

In [20]:
def register_hooks(model):
    trace = {}
    handles = []
    adaln_full = {}

    def hook_layer(name, module):
        def hook(module, inputs, output):
            x = first_tensor(inputs)
            y = first_tensor(output)
            entry = trace.setdefault(name, {})
            if x is not None:
                entry["forward_in"] = stats(x)
                if x.requires_grad:
                    x.register_hook(lambda g, n=name: trace[n].update({"grad_in": stats(g)}))
            if y is not None:
                entry["forward_out"] = stats(y)
                if y.requires_grad:
                    y.register_hook(lambda g, n=name: trace[n].update({"grad_out": stats(g)}))
        handles.append(module.register_forward_hook(hook))

    def hook_adaln(block_idx, module):
        name = f"Block {block_idx}.adaLN"
        def hook(module, inputs, output):
            x = first_tensor(inputs)
            entry = trace.setdefault(name, {})
            entry["forward_in"] = stats(x)
            entry["forward_out"] = stats(output)
            chunks = output.chunk(6, dim=-1)
            names = ["shift_msa", "scale_msa", "gate_msa", "shift_mlp", "scale_mlp", "gate_mlp"]
            entry["components"] = {k: stats(v) for k, v in zip(names, chunks)}
            adaln_full[block_idx] = {"gate_msa": chunks[2].detach(), "gate_mlp": chunks[5].detach()}
            if x.requires_grad:
                x.register_hook(lambda g, n=name: trace[n].update({"grad_in": stats(g)}))
            if output.requires_grad:
                def grad_hook(g):
                    trace[name]["grad_out"] = stats(g)
                    gchunks = g.chunk(6, dim=-1)
                    trace[name]["grad_components"] = {k: stats(v) for k, v in zip(names, gchunks)}
                output.register_hook(grad_hook)
        handles.append(module.register_forward_hook(hook))

    def hook_attn(block_idx, module):
        name = f"Block {block_idx}.attn"
        def hook(module, inputs, output):
            x = first_tensor(inputs)
            y = first_tensor(output)
            entry = trace.setdefault(name, {})
            entry["forward_in"] = stats(x)
            entry["forward_out"] = stats(y)
            gate = adaln_full[block_idx]["gate_msa"]
            entry["residual_branch"] = stats(gate.unsqueeze(1) * y.detach())
            if x.requires_grad:
                x.register_hook(lambda g, n=name: trace[n].update({"grad_in": stats(g)}))
            if y.requires_grad:
                y.register_hook(lambda g, n=name: trace[n].update({"grad_out": stats(g)}))
        handles.append(module.register_forward_hook(hook))

    def hook_mlp(block_idx, module):
        name = f"Block {block_idx}.mlp"
        def hook(module, inputs, output):
            x = first_tensor(inputs)
            y = first_tensor(output)
            entry = trace.setdefault(name, {})
            entry["forward_in"] = stats(x)
            entry["forward_out"] = stats(y)
            gate = adaln_full[block_idx]["gate_mlp"]
            entry["residual_branch"] = stats(gate.unsqueeze(1) * y.detach())
            if x.requires_grad:
                x.register_hook(lambda g, n=name: trace[n].update({"grad_in": stats(g)}))
            if y.requires_grad:
                y.register_hook(lambda g, n=name: trace[n].update({"grad_out": stats(g)}))
        handles.append(module.register_forward_hook(hook))

    hook_layer("PatchEmbed", model.x_embedder)
    for i, block in enumerate(model.blocks):
        hook_layer(f"Block {i}", block)
        hook_adaln(i, block.adaLN_modulation)
        hook_layer(f"Block {i}.norm1", block.norm1)
        hook_layer(f"Block {i}.qkv", block.attn.qkv)
        hook_attn(i, block.attn)
        hook_layer(f"Block {i}.attn_proj", block.attn.proj)
        hook_layer(f"Block {i}.norm2", block.norm2)
        hook_layer(f"Block {i}.fc1", block.mlp.fc1)
        hook_layer(f"Block {i}.fc2", block.mlp.fc2)
        hook_mlp(i, block.mlp)
    hook_layer("FinalLayer", model.final_layer)
    return trace, handles

In [21]:
def run_rank(snapshot, rank_name):
    model = build_model(snapshot)
    trace, handles = register_hooks(model)

    z = move_to(snapshot["z"])
    parents = move_to(snapshot["parents"])
    zs = move_to(snapshot.get("zs"))
    time_input = move_to(snapshot.get("time_input")) if snapshot.get("time_input") is not None else None
    noises = move_to(snapshot.get("noises")) if snapshot.get("noises") is not None else None

    if time_input is None or noises is None:
        restore_rng(snapshot)

    model.zero_grad(set_to_none=True)

    with precision_context(snapshot):
        out = model(
            x=z,
            parents=parents,
            zs=zs,
            loss_kwargs={
                "weighting": getarg("weighting", "uniform"),
                "path_type": getarg("path_type", "linear"),
                "prediction": getarg("prediction", "v"),
                "align_only": False,
            },
            time_input=time_input,
            noises=noises,
        )
        denoise = out["denoising_loss"].mean()
        proj = out["proj_loss"].mean()
        loss = denoise + getarg("proj_coeff", 0.0) * proj

    loss.backward()
    for h in handles:
        h.remove()

    print(f"{rank_name} loss: {loss.item():.8g}")
    print("saved loss:", snapshot.get("losses", {}).get("sit_loss"))
    return model, trace

In [22]:
def ratio(a, b):
    return a / b if b is not None and b > 0 else float("nan")

def fmt(x):
    return "N/A" if x is None or x != x else f"{x:.4g}"

def print_layer(name, t, indent=""):
    fi = t.get("forward_in")
    fo = t.get("forward_out")
    gi = t.get("grad_in")
    go = t.get("grad_out")
    fwd_gain = ratio(fo["rms"], fi["rms"]) if fi is not None and fo is not None else float("nan")
    bwd_gain = ratio(gi["rms"], go["rms"]) if gi is not None and go is not None else float("nan")
    bad = 0
    for s in [fi, fo, gi, go]:
        if s is not None:
            bad += s["nan"] + s["inf"]
    print(
        f"{indent}{name:18s} "
        f"{fmt(fi['rms'] if fi else None):>10s} "
        f"{fmt(fo['rms'] if fo else None):>10s} "
        f"{fmt(fwd_gain):>10s} "
        f"{fmt(go['rms'] if go else None):>10s} "
        f"{fmt(gi['rms'] if gi else None):>10s} "
        f"{fmt(bwd_gain):>10s} "
        f"{fmt(fo['absmax'] if fo else None):>10s} "
        f"{bad:6d}"
    )

def print_deep_trace(trace, title):
    print(f"\n===== {title} =====")
    print(
        f"{'Layer':20s} "
        f"{'In RMS':>10s} {'Out RMS':>10s} {'Fwd gain':>10s} "
        f"{'Grad out':>10s} {'Grad in':>10s} {'Bwd gain':>10s} "
        f"{'Out max':>10s} {'bad':>6s}"
    )
    print_layer("PatchEmbed", trace["PatchEmbed"])
    block_ids = sorted(int(k.split()[1]) for k in trace if k.startswith("Block ") and "." not in k)
    for i in block_ids:
        print(f"\n--- Block {i} ---")
        print_layer("BLOCK", trace[f"Block {i}"], indent="  ")
        print_layer("adaLN", trace[f"Block {i}.adaLN"], indent="  ")
        print_layer("norm1", trace[f"Block {i}.norm1"], indent="  ")
        print_layer("qkv", trace[f"Block {i}.qkv"], indent="  ")
        print_layer("attention", trace[f"Block {i}.attn"], indent="  ")
        print_layer("attn_proj", trace[f"Block {i}.attn_proj"], indent="  ")
        print_layer("norm2", trace[f"Block {i}.norm2"], indent="  ")
        print_layer("fc1", trace[f"Block {i}.fc1"], indent="  ")
        print_layer("fc2", trace[f"Block {i}.fc2"], indent="  ")
        print_layer("mlp", trace[f"Block {i}.mlp"], indent="  ")

        a = trace[f"Block {i}.adaLN"]["components"]
        print("  adaLN components:")
        for name in ["shift_msa", "scale_msa", "gate_msa", "shift_mlp", "scale_mlp", "gate_mlp"]:
            s = a[name]
            print(f"    {name:10s} RMS={s['rms']:10.4g} absmax={s['absmax']:10.4g} bad={s['nan'] + s['inf']}")

        attn_branch = trace[f"Block {i}.attn"]["residual_branch"]
        mlp_branch = trace[f"Block {i}.mlp"]["residual_branch"]
        print(
            f"  residual branches: attn RMS={attn_branch['rms']:.4g}, absmax={attn_branch['absmax']:.4g} | "
            f"mlp RMS={mlp_branch['rms']:.4g}, absmax={mlp_branch['absmax']:.4g}"
        )

    print("\n--- Final ---")
    print_layer("FinalLayer", trace["FinalLayer"], indent="  ")

In [23]:
model0, trace0 = run_rank(rank0, "Rank 0")
print_deep_trace(trace0, "RANK 0")

Creating embedder for parent 'age' with input dim 1 and output dim 96
Creating embedder for parent 'view' with input dim 2 and output dim 96
Creating embedder for parent 'density' with input dim 4 and output dim 96
Creating embedder for parent 'scanner' with input dim 5 and output dim 96
Rank 0 loss: 25.620625
saved loss: 25.620624542236328

===== RANK 0 =====
Layer                    In RMS    Out RMS   Fwd gain   Grad out    Grad in   Bwd gain    Out max    bad
PatchEmbed             0.7854     0.2444     0.3112     0.9505        N/A        N/A      3.172      0

--- Block 0 ---
  BLOCK                  0.7486      314.1      419.6   0.001587     0.9508      599.1       4677      0
  adaLN                   15.83      7.163     0.4526     0.3102      35.35        114       59.5      0
  norm1                  0.7486          1      1.336     0.5438     0.9508      1.749      3.884      0
  qkv                     9.437      7.983     0.8459    0.04453    0.05873      1.319      42.75

In [24]:
del model0
torch.cuda.empty_cache()

model1, trace1 = run_rank(rank1, "Rank 1")
print_deep_trace(trace1, "RANK 1")

Creating embedder for parent 'age' with input dim 1 and output dim 96
Creating embedder for parent 'view' with input dim 2 and output dim 96
Creating embedder for parent 'density' with input dim 4 and output dim 96
Creating embedder for parent 'scanner' with input dim 5 and output dim 96
Rank 1 loss: 24.728966
saved loss: 24.728965759277344

===== RANK 1 =====
Layer                    In RMS    Out RMS   Fwd gain   Grad out    Grad in   Bwd gain    Out max    bad
PatchEmbed             0.8181     0.2558     0.3127      0.324        N/A        N/A      3.094      0

--- Block 0 ---
  BLOCK                  0.7525      300.2      398.9  0.0008669      0.324      373.8       4972      0
  adaLN                   15.39      6.968     0.4528     0.1583      15.11      95.48      60.25      0
  norm1                  0.7525          1      1.329     0.1934      0.324      1.675      4.177      0
  qkv                     9.197       7.78     0.8459     0.0158    0.02084      1.319       45.5